# Calibrated Division Tracking in 3D+Time

Three things jumped out when I spent time with the leading public UNet+ILP approach:

**Detection threshold never validated.** `det_threshold=0.99` is inherited from the reference notebook and applied uniformly to every video — regardless of how dense or sparse that particular embryo is. The metric explicitly penalises over-predicting the total node count, which means a single global threshold is a poor fit when one sample has ~76 cells per frame and another has ~750.

**Divisions nearly absent.** Only 1–6 division events surface across a full 100-frame video of early zebrafish development — a stage where cells divide constantly. Looking at the ILP objective: with `division_weight=1.0` and `edge_weight=-1.0`, a split (one parent → two daughters, two edge rewards) accrues the same net cost as a plain continuation. The optimizer picks division only when the edge classifier is highly confident about *both* daughter links simultaneously. Lowering the division cost gives the model room to commit when evidence is present but just short of certain.

**No validation on training data at all.** The public solution doesn't show what the proxy score looks like on training samples under different settings. I want that before committing to a submission.

What I did: cross-validate `det_threshold` × `division_weight` on a handful of training samples from distinct embryos (same embryo-disjoint structure as the train/test split), pick the combination that best matches the competition's edge + division Jaccard structure, then run the full test set with those settings.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ model -- #
COMPETITION    = "biohub-cell-tracking-during-development"
METHOD         = "unet_transformer"
UNET_BATCH     = 4
ILP_EDGE_W     = -1.0
ILP_APPEAR_W   = 0.1
ILP_DISAPPEAR_W= 0.1

# -------------------------------------------------------------------- CV -- #
# I tried det_threshold=0.80 / div_weight=0.1 as the most aggressive combo.
# It produced a lot of spurious divisions on both CV embryos and hurt the
# edge Jaccard more than the division gain was worth.  The ranges below are
# the safe zone that CV actually distinguishes.
CV_DET_THRESHOLDS = [0.99, 0.95, 0.90]
CV_DIV_WEIGHTS    = [1.0,  0.5]
CV_MAX_EMBRYOS    = 3

BEST_DET_THR = 0.99   # overwritten by CV if it runs
BEST_DIV_W   = 1.0

# ----------------------------------------------------------------- paths -- #
COMP_DIR = next(
    (
        p for p in [
            Path(f"/kaggle/input/competitions/{COMPETITION}"),
            Path(f"/kaggle/input/{COMPETITION}"),
        ]
        if p.exists()
    ),
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
)
TEST_DIR  = COMP_DIR / "test"
TRAIN_DIR = COMP_DIR / "train"
WORKING   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR  = WORKING / "cellmot_repo"
SUB_PATH  = WORKING / "submission.csv"

print(f"COMP_DIR : {COMP_DIR}  (exists={COMP_DIR.exists()})")
print(f"TEST_DIR : {TEST_DIR}  (exists={TEST_DIR.exists()})")
print(f"TRAIN_DIR: {TRAIN_DIR} (exists={TRAIN_DIR.exists()})")

## Offline setup

Everything — wheels, repository source, pretrained weights — comes from the `cellmot-baseline-artifacts` public dataset.  No internet needed.

In [ ]:
def find_artifacts() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/thibautgoldsborough"
             "/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        Path("/kaggle/input/cellmot-baseline-artifacts"),
    ]
    root = Path("/kaggle/input")
    if root.exists():
        for child in root.iterdir():
            if "cellmot" in child.name.lower():
                candidates += [child, child / child.name]
    seen: set[Path] = set()
    for p in candidates:
        p = p.resolve()
        if p in seen:
            continue
        seen.add(p)
        if (p / "repo").exists() and (p / "weights").exists() and (p / "wheels").exists():
            return p
    raise FileNotFoundError(
        "cellmot-baseline-artifacts not found.\n"
        + "Searched: " + str([str(c) for c in list(seen)[:8]])
    )


ARTIFACTS = find_artifacts()
print("ARTIFACTS:", ARTIFACTS)
print("Contents: ", sorted(p.name for p in ARTIFACTS.iterdir()))

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", str(ARTIFACTS / "wheels"),
        "--quiet",
        "tracksdata", "zarr>=3.0.10", "pyscipopt",
    ],
    check=True,
)
print("Packages installed")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)
sys.path.insert(0, str(REPO_DIR / "src"))

weights_root    = REPO_DIR / "weights" / METHOD
AVAILABLE_SPLITS = (
    sorted(d.name for d in weights_root.iterdir() if d.is_dir())
    if weights_root.exists() else ["split_0"]
)
print(f"Available model splits: {AVAILABLE_SPLITS}")
for sp in AVAILABLE_SPLITS:
    wd = REPO_DIR / "weights" / METHOD / sp
    print(f"  {sp}: {sorted(p.name for p in wd.iterdir())}")

## Cross-validation framework

The proxy metric isn't a perfect copy of the competition score — I don't have the exact over-prediction penalty formula — but it captures the same structure:

- Nodes are matched per-frame with a Hungarian assignment gated at 7 µm (physical coordinates, z=1.625 µm/vox, y=x=0.40625 µm/vox).
- Edge Jaccard is computed over the matched subgraph.  When `N_pred > estimated_N_gt`, it's multiplied by `N_est / N_pred` — a conservative stand-in for the official penalty.
- Division Jaccard: nodes with ≥2 outgoing edges are division events.  After mapping predicted divisions through the node-match, TP/FP/FN give a per-sample Jaccard.
- Combined = mean(adj_ej, div_j) — rough approximation, but enough to rank settings.

Training samples are drawn one-per-embryo to keep the CV embryo-diverse, matching the train/test disjointness structure.

In [ ]:
import zarr
import tracksdata as td
from scipy.optimize import linear_sum_assignment

SCALE_UM = np.array([1.625, 0.40625, 0.40625])   # z, y, x µm/voxel
GATE_UM  = 7.0


def list_zarr_names(directory: Path) -> list[str]:
    if not directory.exists():
        return []
    return sorted(p.stem for p in directory.iterdir() if p.suffix == ".zarr")


def read_geff(path: Path):
    """Returns (nodes_df, edges_df, estimated_total_nodes)."""
    g    = zarr.open_group(str(path), mode="r")
    ids  = np.asarray(g["nodes/ids"][:]).astype(np.int64)
    props = {k: np.asarray(g[f"nodes/props/{k}/values"][:]) for k in ("t","z","y","x")}
    nodes = pd.DataFrame({"node_id": ids, **props})

    raw = np.asarray(g["edges/ids"][:]).astype(np.int64)
    edges = (
        pd.DataFrame(raw, columns=["source_id", "target_id"])
        if raw.ndim == 2 and raw.shape[1] == 2
        else pd.DataFrame(columns=["source_id", "target_id"])
    )

    est = np.nan
    try:
        meta = json.loads((path / "zarr.json").read_text())

        def _dig(obj):
            if isinstance(obj, dict):
                if "estimated_number_of_nodes" in obj:
                    return obj["estimated_number_of_nodes"]
                for v in obj.values():
                    r = _dig(v)
                    if r is not None:
                        return r
            return None

        v = _dig(meta)
        if v is not None:
            est = float(v)
    except Exception:
        pass

    return nodes, edges, est


def load_pred_graph(geff_path: Path):
    result = td.graph.IndexedRXGraph.from_geff(str(geff_path))
    return result[0] if isinstance(result, tuple) else result


def graph_to_rows(graph, dataset: str) -> list[dict]:
    rows: list[dict] = []
    for r in graph.node_attrs().iter_rows(named=True):
        rows.append({
            "dataset": dataset, "row_type": "node",
            "node_id": int(r["node_id"]),
            "t": int(r["t"]),
            "z": int(round(float(r["z"]))),
            "y": int(round(float(r["y"]))),
            "x": int(round(float(r["x"]))),
            "source_id": -1, "target_id": -1,
        })
    for r in graph.edge_attrs().iter_rows(named=True):
        rows.append({
            "dataset": dataset, "row_type": "edge",
            "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(r["source_id"]),
            "target_id": int(r["target_id"]),
        })
    return rows


def proxy_score(
    pn: pd.DataFrame,
    pe: pd.DataFrame,
    gn: pd.DataFrame,
    ge: pd.DataFrame,
    n_est: float = np.nan,
) -> dict:
    # --- node matching per timepoint ---
    p2g: dict[int, int] = {}
    for t in sorted(set(pn["t"].unique()) & set(gn["t"].unique())):
        p_t = pn[pn["t"] == t].reset_index(drop=True)
        g_t = gn[gn["t"] == t].reset_index(drop=True)
        if len(p_t) == 0 or len(g_t) == 0:
            continue
        D = np.sqrt(
            ((p_t[["z","y","x"]].values * SCALE_UM)[:, None] -
             (g_t[["z","y","x"]].values * SCALE_UM)[None]) ** 2
        ).sum(2)
        cost = np.where(D <= GATE_UM, D, 1e9)
        ri, ci = linear_sum_assignment(cost)
        for r, c in zip(ri, ci):
            if cost[r, c] < 1e9:
                p2g[int(p_t.loc[r, "node_id"])] = int(g_t.loc[c, "node_id"])

    # --- edge Jaccard ---
    gt_edge_set = set(
        zip(ge["source_id"].astype(int), ge["target_id"].astype(int))
    )
    pm = {
        (p2g[int(s)], p2g[int(t)])
        for s, t in zip(pe["source_id"].astype(int), pe["target_id"].astype(int))
        if int(s) in p2g and int(t) in p2g
    }
    e_tp  = len(pm & gt_edge_set)
    e_fp  = len(pm) - e_tp
    e_fn  = len(gt_edge_set) - e_tp
    raw_ej = e_tp / max(e_tp + e_fp + e_fn, 1)

    n_pred = len(pn)
    penalty = (
        min(1.0, float(n_est) / max(n_pred, 1))
        if np.isfinite(n_est) and n_pred > n_est
        else 1.0
    )
    adj_ej = raw_ej * penalty

    # --- division Jaccard ---
    def _div_nodes(e: pd.DataFrame) -> set[int]:
        if len(e) == 0:
            return set()
        cnt = e.groupby("source_id").size()
        return set(cnt[cnt >= 2].index.astype(int))

    gt_divs  = _div_nodes(ge)
    pd_divs  = _div_nodes(pe)
    pd_divs_mapped = {p2g[n] for n in pd_divs if n in p2g}

    d_tp = len(pd_divs_mapped & gt_divs)
    d_fp = len(pd_divs) - d_tp
    d_fn = len(gt_divs) - d_tp
    div_j = d_tp / max(d_tp + d_fp + d_fn, 1)

    return {
        "n_pred": n_pred, "n_gt": len(gn), "n_est": n_est,
        "n_matched": len(p2g), "penalty": round(penalty, 4),
        "raw_ej": round(raw_ej, 4), "adj_ej": round(adj_ej, 4),
        "n_gt_div": len(gt_divs), "n_pred_div": len(pd_divs),
        "div_tp": d_tp, "div_j": round(div_j, 4),
        "combined": round((adj_ej + div_j) / 2, 4),
    }


print("Utilities ready")

In [ ]:
def pick_cv_samples(train_dir: Path, n: int) -> list[str]:
    names = [
        nm for nm in list_zarr_names(train_dir)
        if (train_dir / (nm + ".geff")).exists()
    ]
    seen: set[str] = set()
    picked: list[str] = []
    for nm in names:
        eid = nm.split("_")[0]
        if eid not in seen:
            seen.add(eid)
            picked.append(nm)
        if len(picked) >= n:
            break
    return picked


cv_samples = pick_cv_samples(TRAIN_DIR, CV_MAX_EMBRYOS)
print(f"CV samples ({len(cv_samples)}): {cv_samples}")

if not cv_samples:
    print("WARNING: no CV samples found — will skip sweep and use defaults.")

In [ ]:
def run_predict(
    data_dir: Path,
    sample_names: list[str],
    det_thr: float,
    div_w: float,
    split_name: str = "split_0",
    tag: str = "run",
) -> list[Path]:
    """Invoke predict_unet_transformer.py; return sorted list of output .geff paths."""
    splits_file = REPO_DIR / f"splits_{tag}.json"
    splits_file.write_text(
        json.dumps([{"split": 0, "train": [], "test": sample_names}])
    )

    # wipe previous predictions so runs don't bleed into each other
    pred_base = REPO_DIR / "predictions"
    if pred_base.exists():
        shutil.rmtree(pred_base)

    cmd = [
        sys.executable, "scripts/predict_unet_transformer.py",
        "--data-dir",                str(data_dir),
        "--splits",                  splits_file.name,
        "--split",                   "0",
        "--weights",                 f"weights/{METHOD}/{split_name}/edge_predictor_best.pth",
        "--unet-batch-size",         str(UNET_BATCH),
        "--det-threshold",           str(det_thr),
        "--ilp-edge-weight",         str(ILP_EDGE_W),
        "--ilp-appearance-weight",   str(ILP_APPEAR_W),
        "--ilp-disappearance-weight",str(ILP_DISAPPEAR_W),
        "--ilp-division-weight",     str(div_w),
        "--use-ilp",
    ]

    result = subprocess.run(
        cmd,
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": "src"},
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Prediction script failed (thr={det_thr}, div_w={div_w}):\n"
            + result.stderr.decode(errors="replace")[:3000]
        )

    found = sorted(pred_base.rglob("*.geff")) if pred_base.exists() else []
    if not found:
        raise RuntimeError(f"No .geff outputs found under {pred_base}")
    return found


print("run_predict helper ready")

In [ ]:
cv_records: list[dict] = []
cv_start = time.time()

if cv_samples:
    combos = list(product(CV_DET_THRESHOLDS, CV_DIV_WEIGHTS))
    print(
        f"{len(combos)} combos × {len(cv_samples)} samples "
        f"= {len(combos)*len(cv_samples)} total predictions"
    )

    for det_thr, div_w in combos:
        print(f"\n>>> det_thr={det_thr}  div_w={div_w}")
        t0 = time.time()
        try:
            pred_geffs = run_predict(
                TRAIN_DIR, cv_samples, det_thr, div_w,
                tag=f"cv_{str(det_thr).replace('.','p')}_{str(div_w).replace('.','p')}",
            )
            combo_c: list[float] = []
            for geff_path in pred_geffs:
                stem   = geff_path.stem
                gt_pth = TRAIN_DIR / (stem + ".geff")
                if not gt_pth.exists():
                    print(f"    [skip] no GT for {stem}")
                    continue
                gn, ge, n_est = read_geff(gt_pth)
                graph = load_pred_graph(geff_path)
                rows  = pd.DataFrame(graph_to_rows(graph, stem))
                pn    = rows[rows["row_type"] == "node"]
                pe    = rows[rows["row_type"] == "edge"]
                sc    = proxy_score(pn, pe, gn, ge, n_est)
                sc.update(dataset=stem, det_thr=det_thr, div_w=div_w)
                cv_records.append(sc)
                combo_c.append(sc["combined"])
                print(
                    f"    {stem[:28]:28s}  "
                    f"adj_ej={sc['adj_ej']:.4f}  "
                    f"div_j={sc['div_j']:.4f}  "
                    f"combined={sc['combined']:.4f}  "
                    f"pred_div={sc['n_pred_div']}  gt_div={sc['n_gt_div']}"
                )
            m = np.mean(combo_c) if combo_c else float("nan")
            print(f"    => mean combined = {m:.4f}  ({time.time()-t0:.0f}s)")
        except Exception as exc:
            print(f"    FAILED: {str(exc)[:250]}")

    print(f"\nCV wall time: {(time.time()-cv_start)/60:.1f} min")
else:
    print("No CV samples — skipping sweep.")

In [ ]:
if cv_records:
    cv_df = pd.DataFrame(cv_records)
    summary = (
        cv_df.groupby(["det_thr", "div_w"])[["adj_ej", "div_j", "combined"]]
        .mean()
        .round(4)
        .sort_values("combined", ascending=False)
    )
    print("Mean proxy scores across CV embryos:")
    display(summary)

    best_idx     = summary.index[0]
    BEST_DET_THR = float(best_idx[0])
    BEST_DIV_W   = float(best_idx[1])

    best_c = float(summary.iloc[0]["combined"])
    base_c = float(
        summary.loc[(0.99, 1.0), "combined"]
        if (0.99, 1.0) in summary.index
        else float("nan")
    )
    print(f"\nBest  : det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}  combined={best_c:.4f}")
    if np.isfinite(base_c):
        print(f"vs defaults (0.99 / 1.0): combined={base_c:.4f}  "
              f"gain={best_c - base_c:+.4f}")
else:
    print(f"No CV data.  Using defaults: det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}")

print(f"\nFinal inference params: det_thr={BEST_DET_THR}  div_w={BEST_DIV_W}")

## Test inference

If more than one model checkpoint exists in the artifact, each gets its own inference pass. At the tracking-graph level a true ensemble is non-trivial — node IDs differ across runs and there's no natural way to average discrete linking decisions — so I collect per-split stats and use the checkpoint from the split with the most confident division signal. If only one split exists, that's the submission.

In [ ]:
test_names = list_zarr_names(TEST_DIR)
print(f"{len(test_names)} test videos: {test_names[:6]}{'...' if len(test_names)>6 else ''}")

split_rows:  dict[str, list[dict]] = {}
split_stats: list[dict] = []
t_infer = time.time()

for split_name in AVAILABLE_SPLITS:
    wp = REPO_DIR / "weights" / METHOD / split_name / "edge_predictor_best.pth"
    if not wp.exists():
        print(f"  [skip] weights not found: {wp}")
        continue

    print(f"\n=== {split_name} ===")
    t0 = time.time()
    try:
        pred_geffs = run_predict(
            TEST_DIR, test_names, BEST_DET_THR, BEST_DIV_W,
            split_name=split_name,
            tag=f"test_{split_name}",
        )
    except Exception as exc:
        print(f"  FAILED: {exc}")
        continue

    rows: list[dict] = []
    for geff_path in pred_geffs:
        graph = load_pred_graph(geff_path)
        rows.extend(graph_to_rows(graph, geff_path.stem))

    df      = pd.DataFrame(rows)
    n_nodes = int((df["row_type"] == "node").sum())
    n_edges = int((df["row_type"] == "edge").sum())
    e_df    = df[df["row_type"] == "edge"]
    n_div   = int((e_df.groupby("source_id").size() >= 2).sum()) if len(e_df) else 0
    elapsed = time.time() - t0

    print(f"  nodes={n_nodes:,}  edges={n_edges:,}  divisions={n_div}  ({elapsed/60:.1f} min)")
    split_rows[split_name]  = rows
    split_stats.append(dict(
        split=split_name, nodes=n_nodes, edges=n_edges,
        divisions=n_div, minutes=round(elapsed/60, 2),
    ))

print(f"\nTotal inference: {(time.time()-t_infer)/60:.1f} min")
if split_stats:
    display(pd.DataFrame(split_stats))

In [ ]:
# choose the canonical split: prefer the one with the most divisions detected
# (assuming division recall is the bigger gap vs. the baseline)
if len(split_stats) > 1:
    best_sp = max(split_stats, key=lambda s: s["divisions"])["split"]
    print(f"Multiple splits; selecting {best_sp} (most divisions detected)")
elif split_stats:
    best_sp = split_stats[0]["split"]
    print(f"Single split: {best_sp}")
else:
    raise RuntimeError("No successful test inference run.")

SUBMISSION_COLS = [
    "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
]

submission = pd.DataFrame(split_rows[best_sp], columns=SUBMISSION_COLS)
submission.index.name = "id"

# ---------------------------------------------------------------- checks -- #
pred_ds   = set(submission["dataset"].unique())
missing   = sorted(set(test_names) - pred_ds)
if missing:
    raise AssertionError(f"Missing datasets: {missing}")

nodes = submission[submission["row_type"] == "node"]
edges = submission[submission["row_type"] == "edge"]

assert len(nodes) > 0, "No node rows"
assert not submission.isna().any().any(), "NaN in submission"
assert set(submission["row_type"].unique()).issubset({"node", "edge"})
assert (nodes[["node_id","t","z","y","x"]] >= 0).all().all()
assert (nodes[["source_id","target_id"]] == -1).all().all()
if len(edges):
    assert (edges[["node_id","t","z","y","x"]] == -1).all().all()
    assert (edges[["source_id","target_id"]] >= 0).all().all()

for ds, g in submission.groupby("dataset"):
    dn = g[g["row_type"] == "node"]
    de = g[g["row_type"] == "edge"]
    nids = set(dn["node_id"])
    assert dn["node_id"].is_unique,           f"Duplicate node_id in {ds}"
    assert de["source_id"].isin(nids).all(),  f"Dangling source_id in {ds}"
    assert de["target_id"].isin(nids).all(),  f"Dangling target_id in {ds}"

submission.to_csv(SUB_PATH)
print(f"Wrote {SUB_PATH}")
print(f"  {len(submission):,} rows | {len(nodes):,} nodes | {len(edges):,} edges")

# per-dataset breakdown
ds_stats: list[dict] = []
for ds, g in submission.groupby("dataset"):
    gn = g[g["row_type"] == "node"]
    ge = g[g["row_type"] == "edge"]
    od = ge.groupby("source_id").size() if len(ge) else pd.Series(dtype=int)
    ds_stats.append(dict(
        dataset=ds,
        frames=int(gn["t"].nunique()),
        nodes=len(gn),
        edges=len(ge),
        cells_per_frame=round(len(gn) / max(gn["t"].nunique(), 1), 1),
        divisions=int((od >= 2).sum()),
    ))

display(pd.DataFrame(ds_stats))
display(submission.head())